In [1]:
from pathlib import Path
import zipfile
import tempfile

zip_path = Path("/content/IDX_Analysis 2.zip")

if not zip_path.is_file():
    raise FileNotFoundError(f"Please upload: {zip_path.name}")

if not zipfile.is_zipfile(zip_path):
    raise ValueError("The ZIP is invalid or the upload is incomplete.")

# Use a fresh folder to avoid mixing files from earlier extractions.
data_folder = Path(tempfile.mkdtemp(prefix="ds_preprocessing_", dir="/content"))

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(data_folder)

csv_files = sorted(
    path for path in data_folder.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".csv"
    and "__MACOSX" not in path.parts
    and not path.name.startswith(".")
)

sold_files = [
    path for path in csv_files
    if path.name.startswith("CRMLSSold")
]

print("Extracted to:", data_folder)
print("Total CSV files:", len(csv_files))
print("Sold CSV files:", len(sold_files))

Extracted to: /content/ds_preprocessing_e6metpj8
Total CSV files: 66
Sold CSV files: 33


In [2]:
import pandas as pd
import re

# Match monthly sold filenames, including optional _filled versions.
by_month = {}

for path in sold_files:
    match = re.fullmatch(
        r"CRMLSSold(\d{6})(?:_filled)?\.csv",
        path.name,
        flags=re.IGNORECASE
    )

    if match:
        month = match.group(1)
        by_month.setdefault(month, []).append(path)

expected_months = pd.period_range(
    "2024-01", "2026-09", freq="M"
).strftime("%Y%m")

missing = [month for month in expected_months if month not in by_month]
duplicates = {
    month: [path.name for path in paths]
    for month, paths in by_month.items()
    if len(paths) > 1
}

if missing or duplicates:
    raise ValueError(
        f"Missing months: {missing}\n"
        f"Multiple files for the same month: {duplicates}"
    )

required_columns = [
    "ListingKey",
    "CloseDate",
    "PropertyType",
    "PropertySubType",
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "City",
    "CountyOrParish",
    "PostalCode",
    "StateOrProvince"
]

frames = []
total_raw = 0

for month in expected_months:
    path = by_month[month][0]

    monthly = pd.read_csv(
        path,
        usecols=required_columns,
        dtype={"PostalCode": "string", "ListingKey": "string"},
        low_memory=False
    )

    total_raw += len(monthly)

    # Apply the Data Science project's required property filters.
    monthly = monthly.loc[
        monthly["PropertyType"].eq("Residential")
        & monthly["PropertySubType"].eq("SingleFamilyResidence")
    ].copy()

    frames.append(monthly)

sales = pd.concat(frames, ignore_index=True)
del frames, monthly

# Preserve CloseDate and create a separate parsed date column.
sales["close_date_parsed"] = pd.to_datetime(
    sales["CloseDate"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

sales["sale_month"] = (
    sales["close_date_parsed"].dt.to_period("M")
)

print("Monthly files loaded:", len(expected_months))
print(f"Raw rows read: {total_raw:,}")
print(f"Residential single-family rows: {len(sales):,}")
print("Earliest close date:", sales["close_date_parsed"].min())
print("Latest close date:", sales["close_date_parsed"].max())
print(
    "Missing or unparseable close dates:",
    sales["close_date_parsed"].isna().sum()
)

print("\nRecords by actual close month:")
print(sales["sale_month"].value_counts().sort_index().to_string())

Monthly files loaded: 33
Raw rows read: 736,473
Residential single-family rows: 371,215
Earliest close date: 2024-01-01 00:00:00
Latest close date: 2026-09-30 00:00:00
Missing or unparseable close dates: 0

Records by actual close month:
sale_month
2024-01     8322
2024-02     9564
2024-03    11634
2024-04    12745
2024-05    13831
2024-06    12545
2024-07    13378
2024-08    12433
2024-09    10909
2024-10    12347
2024-11    10692
2024-12    10624
2025-01     8144
2025-02     8851
2025-03    10610
2025-04    11880
2025-05    11777
2025-06    11701
2025-07    12114
2025-08    11454
2025-09    11456
2025-10    12029
2025-11     9739
2025-12    10455
2026-01     7490
2026-02     8959
2026-03    11562
2026-04    12490
2026-05    12466
2026-06    13508
2026-07    12872
2026-08    11448
2026-09    11186
Freq: M


In [3]:
# Initial training window; we will tune this later.
TRAIN_WINDOW_MONTHS = 24

test_month = sales["sale_month"].max()
train_end = test_month - 1
train_start = test_month - TRAIN_WINDOW_MONTHS

train = sales.loc[
    sales["sale_month"].between(train_start, train_end)
].copy()

test = sales.loc[
    sales["sale_month"].eq(test_month)
].copy()

# Confirm every required training month is present.
expected_train_months = pd.period_range(
    train_start, train_end, freq="M"
)

missing_months = expected_train_months.difference(
    pd.PeriodIndex(train["sale_month"].unique(), freq="M")
)

if len(missing_months):
    raise ValueError(f"Missing training months: {list(missing_months)}")

if train.empty or test.empty:
    raise ValueError("The training or test dataset is empty.")

assert train["close_date_parsed"].max() < test["close_date_parsed"].min()

print(f"Training period: {train_start} through {train_end}")
print(f"Training months: {TRAIN_WINDOW_MONTHS}")
print(f"Training rows: {len(train):,}")

print(f"\nTest month: {test_month}")
print(f"Test rows: {len(test):,}")

print(
    f"\nEarlier records outside this training window: "
    f"{len(sales) - len(train) - len(test):,}"
)
print("Earlier records remain available in sales for window experiments.")

Training period: 2024-09 through 2026-08
Training months: 24
Training rows: 265,577

Test month: 2026-09
Test rows: 11,186

Earlier records outside this training window: 94,452
Earlier records remain available in sales for window experiments.


In [4]:
import numpy as np

numeric_columns = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

review = []

for name, dataset in [("Training", train), ("Test", test)]:
    for column in numeric_columns:
        original = dataset[column]

        # Convert numeric text to numbers; unparseable values become NaN.
        converted = pd.to_numeric(original, errors="coerce")

        nonnumeric_count = (
            original.notna() & converted.isna()
        ).sum()

        # Treat infinity as missing.
        infinite_count = converted.isin([np.inf, -np.inf]).sum()
        dataset[column] = converted.replace(
            [np.inf, -np.inf], np.nan
        )

        values = dataset[column]

        # Basic validity rules; extreme positive values remain for review.
        if column in ["ClosePrice", "LivingArea", "LotSizeSquareFeet"]:
            invalid = values.le(0)
        elif column in ["BedroomsTotal", "BathroomsTotalInteger"]:
            invalid = values.lt(0)
        else:
            # Year must be positive and no later than the sale year.
            invalid = (
                values.le(0)
                | values.gt(dataset["close_date_parsed"].dt.year)
            )

        review.append({
            "Dataset": name,
            "Column": column,
            "Missing_count": int(values.isna().sum()),
            "Missing_percent": round(values.isna().mean() * 100, 2),
            "Nonnumeric_count": int(nonnumeric_count),
            "Infinite_count": int(infinite_count),
            "Invalid_count": int(invalid.sum())
        })

numeric_review = pd.DataFrame(review)

display(numeric_review)

,Dataset,Column,Missing_count,Missing_percent,Nonnumeric_count,Infinite_count,Invalid_count
0,Training,ClosePrice,1,0.00,0,0,1
1,Training,LivingArea,140,0.05,0,0,108
2,Training,BedroomsTotal,0,0.00,0,0,0
3,Training,BathroomsTotalInteger,26,0.01,0,0,0
4,Training,LotSizeSquareFeet,4594,1.73,0,0,142
5,Training,YearBuilt,199,0.07,0,0,28
6,Test,ClosePrice,0,0.00,0,0,0
7,Test,LivingArea,3,0.03,0,0,3
8,Test,BedroomsTotal,0,0.00,0,0,0
9,Test,BathroomsTotalInteger,0,0.00,0,0,0


In [5]:
# Work on copies so train and test remain available for comparison.
train_clean = train.copy()
test_clean = test.copy()

cleaning_summary = []

predictor_columns = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

for name, dataset in [
    ("Training", train_clean),
    ("Test", test_clean)
]:
    rows_before = len(dataset)

    # A usable target must be present, finite, and positive.
    # Do not impute the target or remove high prices just for being extreme.
    valid_target = (
        dataset["ClosePrice"].notna()
        & np.isfinite(dataset["ClosePrice"])
        & dataset["ClosePrice"].gt(0)
    )

    dataset.drop(
        index=dataset.index[~valid_target],
        inplace=True
    )

    for column in predictor_columns:
        values = dataset[column]

        if column in ["LivingArea", "LotSizeSquareFeet"]:
            invalid = values.le(0)

        elif column in ["BedroomsTotal", "BathroomsTotalInteger"]:
            invalid = values.lt(0)

        else:
            invalid = (
                values.le(0)
                | values.gt(dataset["close_date_parsed"].dt.year)
            )

        # True means the recorded value failed our validity rule.
        dataset[f"{column}_invalid_flag"] = invalid.fillna(False)

        # Replace invalid predictors with missing values.
        dataset.loc[invalid.fillna(False), column] = np.nan

        # True means this value needs imputation, for either reason:
        # originally missing or marked invalid above.
        dataset[f"{column}_missing_flag"] = dataset[column].isna()

    cleaning_summary.append({
        "Dataset": name,
        "Rows_before": rows_before,
        "Unusable_target_rows_removed": rows_before - len(dataset),
        "Rows_retained": len(dataset)
    })

display(pd.DataFrame(cleaning_summary))

print("\nPredictor values needing imputation:")
display(pd.DataFrame({
    "Training": train_clean[predictor_columns].isna().sum(),
    "Test": test_clean[predictor_columns].isna().sum()
}))

,Dataset,Rows_before,Unusable_target_rows_removed,Rows_retained
0,Training,265577,2,265575
1,Test,11186,0,11186



Predictor values needing imputation:


,Training,Test
LivingArea,248,6
BedroomsTotal,0,0
BathroomsTotalInteger,26,0
LotSizeSquareFeet,4736,198
YearBuilt,227,9


In [6]:
# Calculate replacement values from TRAINING data only.
training_medians = train_clean[predictor_columns].median()

if training_medians.isna().any():
    raise ValueError(
        "Cannot calculate a median for: "
        f"{training_medians.index[training_medians.isna()].tolist()}"
    )

# Preserve the datasets from before imputation.
train_prepared = train_clean.copy()
test_prepared = test_clean.copy()

# Apply the SAME training medians to both datasets.
train_prepared[predictor_columns] = (
    train_prepared[predictor_columns].fillna(training_medians)
)

test_prepared[predictor_columns] = (
    test_prepared[predictor_columns].fillna(training_medians)
)

print("Replacement values learned from training:")
display(training_medians.to_frame("Training_median"))

print("\nMissing predictor values after imputation:")
display(pd.DataFrame({
    "Training": train_prepared[predictor_columns].isna().sum(),
    "Test": test_prepared[predictor_columns].isna().sum()
}))

assert len(train_prepared) == len(train_clean)
assert len(test_prepared) == len(test_clean)
assert train_prepared[predictor_columns].notna().all().all()
assert test_prepared[predictor_columns].notna().all().all()

print("\nPASS: Numeric predictors filled; row counts unchanged.")

Replacement values learned from training:


,Training_median
LivingArea,1813.0
BedroomsTotal,3.0
BathroomsTotalInteger,2.0
LotSizeSquareFeet,7260.0
YearBuilt,1976.0



Missing predictor values after imputation:


,Training,Test
LivingArea,0,0
BedroomsTotal,0,0
BathroomsTotalInteger,0,0
LotSizeSquareFeet,0,0
YearBuilt,0,0



PASS: Numeric predictors filled; row counts unchanged.


In [7]:
categorical_columns = [
    "City",
    "CountyOrParish",
    "PostalCode"
]

for dataset in [train_prepared, test_prepared]:
    for column in categorical_columns:
        values = dataset[column].astype("string").str.strip()

        # Treat blank text as missing.
        values = values.replace("", pd.NA)

        if column == "PostalCode":
            # Remove a trailing .0 from ZIP codes imported as numbers.
            values = values.str.replace(r"\.0$", "", regex=True)
        else:
            # Standardize capitalization.
            values = values.str.upper()

        dataset[column] = values.fillna("Missing")

category_summary = pd.DataFrame({
    "Training_unique_categories": (
        train_prepared[categorical_columns].nunique()
    ),
    "Test_unique_categories": (
        test_prepared[categorical_columns].nunique()
    ),
    "Training_missing": [
        train_prepared[col].eq("Missing").sum()
        for col in categorical_columns
    ],
    "Test_missing": [
        test_prepared[col].eq("Missing").sum()
        for col in categorical_columns
    ]
})

display(category_summary)

,Training_unique_categories,Test_unique_categories,Training_missing,Test_missing
City,1094,671,145,1
CountyOrParish,61,53,0,0
PostalCode,2945,1120,3,0


In [8]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

# Learn categories from training data and encode it.
train_categories_encoded = encoder.fit_transform(
    train_prepared[categorical_columns]
)

# Apply the same encoder to the test data.
test_categories_encoded = encoder.transform(
    test_prepared[categorical_columns]
)

print("Encoded training shape:", train_categories_encoded.shape)
print("Encoded test shape:", test_categories_encoded.shape)

# Count test records containing categories not seen in training.
unseen_summary = []

for column, known_categories in zip(
    categorical_columns, encoder.categories_
):
    unseen = ~test_prepared[column].isin(known_categories)

    unseen_summary.append({
        "Column": column,
        "Test_records_with_unseen_category": int(unseen.sum())
    })

display(pd.DataFrame(unseen_summary))

assert (
    train_categories_encoded.shape[1]
    == test_categories_encoded.shape[1]
)

print("\nPASS: Training and test use the same encoded columns.")

Encoded training shape: (265575, 4100)
Encoded test shape: (11186, 4100)


,Column,Test_records_with_unseen_category
0,City,7
1,CountyOrParish,0
2,PostalCode,71



PASS: Training and test use the same encoded columns.


In [9]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix, hstack

scaler = StandardScaler()

train_numeric_scaled = scaler.fit_transform(
    train_prepared[predictor_columns]
)

test_numeric_scaled = scaler.transform(
    test_prepared[predictor_columns]
)

# Keep the missing/invalid indicators as 0/1; do not scale them.
flag_columns = [
    f"{column}_{suffix}"
    for column in predictor_columns
    for suffix in ["invalid_flag", "missing_flag"]
]

train_flags = train_prepared[flag_columns].to_numpy(dtype=np.float32)
test_flags = test_prepared[flag_columns].to_numpy(dtype=np.float32)

# Combine numeric predictors, flags, and encoded categories.
X_train = hstack([
    csr_matrix(train_numeric_scaled, dtype=np.float32),
    csr_matrix(train_flags),
    train_categories_encoded
], format="csr")

X_test = hstack([
    csr_matrix(test_numeric_scaled, dtype=np.float32),
    csr_matrix(test_flags),
    test_categories_encoded
], format="csr")

# Keep the sale price separate: it is the prediction target.
y_train = train_prepared["ClosePrice"].copy()
y_test = test_prepared["ClosePrice"].copy()

feature_names = (
    predictor_columns
    + flag_columns
    + encoder.get_feature_names_out(categorical_columns).tolist()
)

assert X_train.shape[1] == X_test.shape[1] == len(feature_names)
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)
assert np.isfinite(X_train.data).all()
assert np.isfinite(X_test.data).all()

print("Final training feature shape:", X_train.shape)
print("Final test feature shape:", X_test.shape)
print("Training targets:", len(y_train))
print("Test targets:", len(y_test))
print("\nPASS: Features prepared; no model trained yet.")

Final training feature shape: (265575, 4115)
Final test feature shape: (11186, 4115)
Training targets: 265575
Test targets: 11186

PASS: Features prepared; no model trained yet.


In [10]:
from pathlib import Path
from scipy.sparse import save_npz
import joblib
import json

output_folder = Path("/content/week3_preprocessing_output")
output_folder.mkdir(parents=True, exist_ok=True)

# Readable CSVs: imputed numeric values, categories, and quality flags.
train_prepared.to_csv(
    output_folder / "train_prepared_24months.csv",
    index=False
)

test_prepared.to_csv(
    output_folder / "test_prepared_202609.csv",
    index=False
)

# Model inputs: save sparse matrices without converting to dense CSVs.
save_npz(output_folder / "X_train_24months.npz", X_train)
save_npz(output_folder / "X_test_202609.npz", X_test)

# Targets follow exactly the same row order as the matrices.
y_train.to_csv(output_folder / "y_train_24months.csv", index=False)
y_test.to_csv(output_folder / "y_test_202609.csv", index=False)

# Save the rules learned from training for reuse.
joblib.dump(
    {
        "training_medians": training_medians,
        "encoder": encoder,
        "scaler": scaler,
        "predictor_columns": predictor_columns,
        "categorical_columns": categorical_columns,
        "flag_columns": flag_columns
    },
    output_folder / "preprocessing_24months.joblib"
)

(output_folder / "feature_names.json").write_text(
    json.dumps(feature_names, indent=2),
    encoding="utf-8"
)

print("Saved to:", output_folder)
print("This is the initial 24-month version; window tuning is pending.")


Saved to: /content/week3_preprocessing_output
This is the initial 24-month version; window tuning is pending.


In [11]:
WINDOW_OPTIONS = [6, 12, 18, 24]

validation_months = pd.period_range(
    test_month - 3,
    test_month - 1,
    freq="M"
)

experiment_plan = []

for window in WINDOW_OPTIONS:
    for validation_month in validation_months:
        start_month = validation_month - window
        end_month = validation_month - 1

        training_mask = sales["sale_month"].between(
            start_month, end_month
        )
        validation_mask = sales["sale_month"].eq(validation_month)

        expected = pd.period_range(start_month, end_month, freq="M")
        available = pd.PeriodIndex(
            sales.loc[training_mask, "sale_month"].unique(),
            freq="M"
        )

        missing = expected.difference(available)

        if len(missing):
            raise ValueError(
                f"Missing months for {window}-month window: "
                f"{list(missing)}"
            )

        assert validation_month < test_month

        experiment_plan.append({
            "Training_months": window,
            "Training_start": str(start_month),
            "Training_end": str(end_month),
            "Validation_month": str(validation_month),
            "Training_rows_before_cleaning": int(training_mask.sum()),
            "Validation_rows_before_cleaning": int(validation_mask.sum())
        })

experiment_plan = pd.DataFrame(experiment_plan)
display(experiment_plan)

print(f"\nFinal test month reserved: {test_month}")
print("Experiments planned:", len(experiment_plan))

,Training_months,Training_start,Training_end,Validation_month,Training_rows_before_cleaning,Validation_rows_before_cleaning
0,6,2025-12,2026-05,2026-06,63422,13508
1,6,2026-01,2026-06,2026-07,66475,12872
2,6,2026-02,2026-07,2026-08,71857,11448
3,12,2025-06,2026-05,2026-06,131915,13508
4,12,2025-07,2026-06,2026-07,133722,12872
5,12,2025-08,2026-07,2026-08,134480,11448
6,18,2024-12,2026-05,2026-06,193801,13508
7,18,2025-01,2026-06,2026-07,196685,12872
8,18,2025-02,2026-07,2026-08,201413,11448
9,24,2024-06,2026-05,2026-06,266105,13508



Final test month reserved: 2026-09
Experiments planned: 12


In [12]:
def clean_for_experiment(data):
    """Apply fixed cleaning rules without learning replacement values."""
    cleaned = data.copy()
    rows_before = len(cleaned)

    # Convert numeric fields and replace infinity with missing values.
    for column in ["ClosePrice"] + predictor_columns:
        cleaned[column] = pd.to_numeric(
            cleaned[column], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

    # Exclude records without a usable prediction target.
    usable_target = (
        cleaned["ClosePrice"].notna()
        & cleaned["ClosePrice"].gt(0)
    )
    cleaned = cleaned.loc[usable_target].copy()

    # Flag invalid predictors and turn them into missing values.
    for column in predictor_columns:
        values = cleaned[column]

        if column in ["LivingArea", "LotSizeSquareFeet"]:
            invalid = values.le(0)
        elif column in ["BedroomsTotal", "BathroomsTotalInteger"]:
            invalid = values.lt(0)
        else:
            invalid = (
                values.le(0)
                | values.gt(cleaned["close_date_parsed"].dt.year)
            )

        invalid = invalid.fillna(False)

        cleaned[f"{column}_invalid_flag"] = invalid
        cleaned.loc[invalid, column] = np.nan
        cleaned[f"{column}_missing_flag"] = cleaned[column].isna()

    # Standardize categories using the same fixed rules.
    for column in categorical_columns:
        values = (
            cleaned[column]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

        if column == "PostalCode":
            values = values.str.replace(r"\.0$", "", regex=True)
        else:
            values = values.str.upper()

        cleaned[column] = values.fillna("Missing")

    removed = rows_before - len(cleaned)
    return cleaned, removed


print("Cleaning function ready.")
print("No datasets changed and no models trained in this cell.")

Cleaning function ready.
No datasets changed and no models trained in this cell.


In [13]:
import shutil
from google.colab import files

zip_file = shutil.make_archive(
    "/content/week3_preprocessing_output",
    "zip",
    "/content/week3_preprocessing_output"
)

files.download(zip_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>